# 11 - Spatial delay patterns: distance from Center City, direction, and Amtrak track ownership

Track ownership + distance regression across the full 2017-2025 GTFS history, using the stop id
crosswalk from `02c_gtfs_stops_crosswalk.ipynb` to build a full history that's stable across SEPTA's
stop-id renumbering.

Two findings:
1. Delay increases with distance from Center City, and is much higher Outbound than Inbound.
2. Amtrak-owned track runs significantly later than SEPTA-owned track, also concentrated Outbound and
nearly vanishing Inbound (distance and line fixed effects controlled).

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from shapely.geometry import LineString
import statsmodels.api as sm

from utils import remap_to_canonical_stop_id

BASEPATH = "../data"
START_DATE = "2017-01-01"
END_DATE = "2026-01-01"
CC_TRUNK_STOPS = ["90004", "90005", "90006", "90007", "90008", "90009"]
PROJECTED_CRS = "EPSG:32618"

## 1. Line-direction trips, station geometry, and Center-City-referenced stop position

Derives each line's canonical stop order per direction from the most recent GTFS release, then
normalizes so 0 = at the Center City trunk for both directions -- same style as
`07_reduce_for_modeling.ipynb`'s `stops_from_cc` (Inbound gets flipped onto the same scale as Outbound).

In [2]:
# load station coords + gtfs stops, pick one trip per
# line/direction to set stop order + travel direction
stn_rider = pd.read_csv(BASEPATH + "/4_ridership_by_station.csv")
stn_rider["stop_id"] = stn_rider["stop_id"].astype(str)
stn_points = gpd.GeoDataFrame(
    stn_rider.drop_duplicates("stop_id").copy(),
    geometry = gpd.points_from_xy(
        stn_rider.drop_duplicates("stop_id")["longitude"],
        stn_rider.drop_duplicates("stop_id")["latitude"],
    ),
    crs = "EPSG:4326",
).to_crs(PROJECTED_CRS).set_index("stop_id")
stn_name_lookup = (
    stn_rider.drop_duplicates("stop_id").set_index("stop_id")["station"]
)

# average location of the trunk stations -- reference point for out/inbound
cc_pts = stn_points.loc[stn_points.index.intersection(CC_TRUNK_STOPS)]
cc_centroid_x = cc_pts.geometry.x.mean()
cc_centroid_y = cc_pts.geometry.y.mean()

crosswalk = pd.read_parquet(
    BASEPATH + "/2_gtfs_linkages_since_2017_clean.parquet"
)
crosswalk["line"] = crosswalk["line"].str.replace(" Line$", "", regex = True)
crosswalk = crosswalk.rename(
    columns = {"gtfs_date": "source_gtfs_date"}
)

stop_times_all = pd.read_parquet(BASEPATH + "/2_gtfs_stop_times.parquet")
stop_times_all["stop_sequence"] = stop_times_all["stop_sequence"].astype(int)
stop_times_all = stop_times_all.rename(
    columns = {"gtfs_date": "source_gtfs_date"}
)

# remap stop_times_all to current release stop ids
stop_id_crosswalk = pd.read_parquet(BASEPATH + "/2_stop_id_crosswalk.parquet")
stop_times_all = remap_to_canonical_stop_id(
    stop_times_all, stop_id_crosswalk,
    stop_id_col = "stop_id", date_col = "source_gtfs_date",
)

# use the most recent gtfs release to define each line's stop order
recent_date = crosswalk["source_gtfs_date"].max()
recent_trips = (
    crosswalk[crosswalk["source_gtfs_date"] == recent_date]
    [["line", "trip_id", "source_gtfs_date", "direction_id"]]
    .drop_duplicates()
    .merge(
        stop_times_all,
        on = ["trip_id", "source_gtfs_date"], how = "inner",
    )
)
trip_lengths = (
    recent_trips.groupby(["line", "trip_id", "direction_id"])
    ["stop_sequence"].count()
    .reset_index(name = "n_stops")
)
best_trip_per_dir = (
    trip_lengths.sort_values("n_stops", ascending = False)
    .drop_duplicates(["line", "direction_id"])
)

def orient_trip(trip_id):
    # returns a trip's direction + stops in travel order --
    # direction is whichever end is farther from center city
    ordered = (
        recent_trips[recent_trips["trip_id"] == trip_id]
        .sort_values("stop_sequence")["stop_id"]
        .tolist()
    )
    ordered = [s for s in ordered if s in stn_points.index]
    if len(ordered) < 2:
        return None, None
    first_pt = stn_points.loc[ordered[0], "geometry"]
    last_pt = stn_points.loc[ordered[-1], "geometry"]
    dist_first = (
        (first_pt.x - cc_centroid_x) ** 2
        + (first_pt.y - cc_centroid_y) ** 2
    )
    dist_last = (
        (last_pt.x - cc_centroid_x) ** 2
        + (last_pt.y - cc_centroid_y) ** 2
    )
    return ("Outbound" if dist_first < dist_last else "Inbound"), ordered

# one representative (longest) trip per line/direction
line_dir_rows = []
for _, r in best_trip_per_dir.iterrows():
    real_direction, ordered = orient_trip(r["trip_id"])
    if ordered is None:
        continue
    line_dir_rows.append({
        "line": r["line"],
        "direction_id": r["direction_id"],
        "trip_id": r["trip_id"],
        "real_direction": real_direction,
        "ordered_stops": ordered,
        "n_stops": len(ordered),
    })
line_dirs = pd.DataFrame(line_dir_rows)
print(f"{len(line_dirs)} canonical (line, direction) trips")

remap_to_canonical_stop_id: 5,392,855 rows in, 5,392,193 out (882 with no crosswalk entry, 882 flagged unmatched)
26 canonical (line, direction) trips


In [3]:
# canonical stop position from Center City, same style as
# 07_reduce_for_modeling.ipynb's stops_from_cc: raw index per
# direction, then flip Inbound so 0 = at the trunk for both
stop_dir_rows = []
for _, r in line_dirs.iterrows():
    for i, s in enumerate(r["ordered_stops"]):
        stop_dir_rows.append({
            "line": r["line"], "real_direction": r["real_direction"],
            "stop_position": i, "stop_id": s,
        })
stop_positions = pd.DataFrame(stop_dir_rows)
max_pos = stop_positions.groupby(
    ["line", "real_direction"]
)["stop_position"].transform("max")
stop_positions["stops_from_cc"] = np.where(
    stop_positions["real_direction"] == "Outbound",
    stop_positions["stop_position"],
    max_pos - stop_positions["stop_position"],
)

# Outbound and Inbound should agree on a shared stop's position --
# prefer Outbound's value, fall back to Inbound's for anything
# Outbound's own trip happens to skip
stop_positions["_dir_priority"] = (
    stop_positions["real_direction"] != "Outbound"
).astype(int)
stop_position_lookup = (
    stop_positions.sort_values("_dir_priority")
    .drop_duplicates(["line", "stop_id"])
    .set_index(["line", "stop_id"])["stops_from_cc"]
    .to_dict()
)
print(f"{len(stop_position_lookup)} (line, stop) positions resolved")

214 (line, stop) positions resolved


## 2. Amtrak ownership flags

Ownership boundaries from `dats5990_stop_geography` (originally from google search because GTFS has no
ownership field). Computed once per line using each line's outbound trip.

In [4]:
AMTRAK_SEGMENTS = {
    "Paoli/Thorndale": ("30th Street Station", None),
    "Trenton": ("30th Street Station", None),
    "Wilmington/Newark": ("30th Street Station", None),
    "Airport": ("30th Street Station", "Penn Medicine"),
    "Media/Wawa": ("30th Street Station", "Penn Medicine"),
    "Chestnut Hill West": ("30th Street Station", "North Philadelphia"),
}

# flag each (line, stop) as amtrak-owned, using each line's
# outbound trip to find where the segment starts/ends
amtrak_stop_flags = {}
for _, r in line_dirs[line_dirs["real_direction"] == "Outbound"].iterrows():
    line, ordered_stops = r["line"], r["ordered_stops"]
    names = [stn_name_lookup.get(s, s) for s in ordered_stops]
    if line not in AMTRAK_SEGMENTS:
        for s in ordered_stops:
            amtrak_stop_flags[(line, s)] = False
        continue
    # everything between the segment's start/end
    # station names counts as amtrak-owned
    start_name, end_name = AMTRAK_SEGMENTS[line]
    start_idx = names.index(start_name) if start_name in names else None
    if end_name and end_name in names:
        end_idx = names.index(end_name)
    else:
        end_idx = len(names) - 1
    for i, s in enumerate(ordered_stops):
        amtrak_stop_flags[(line, s)] = (
            start_idx is not None and start_idx <= i <= end_idx
        )

n_amtrak_stops = sum(amtrak_stop_flags.values())
print(f"{n_amtrak_stops} / {len(amtrak_stop_flags)} Amtrak (line, stop)s")

59 / 213 Amtrak (line, stop)s


## 3. Row-level LOCF exposure table

SEPTA's OTP feed only writes a new row when lateness changes so no new rows means lateness stayed constant. `merge_asof(..., direction="backward")` against every
scheduled stop reconstructs the reported lateness at every point on route

In [5]:
# load row-level otp pings, figure out each one's actual scheduled 
# time from its reported lateness
READ_COLS = [
    "service_date", "datetime", "train_number", "lateness",
    "line", "trip_id", "direction_id", "source_gtfs_date",
]
pings_exp = pd.read_parquet(
    BASEPATH + "/2_df_gtfs_linked.parquet", columns = READ_COLS,
    filters = [("service_date", ">=", pd.Timestamp(START_DATE)),
               ("service_date", "<", pd.Timestamp(END_DATE))],
)
pings_exp = pings_exp.dropna(
    subset = ["trip_id", "source_gtfs_date", "line"]
).copy()
pings_exp["train_number"] = pings_exp["train_number"].astype(str)

pings_exp["_time_adj"] = (
    pings_exp["datetime"]
    - pings_exp["service_date"].dt.tz_localize(
        pings_exp["datetime"].dt.tz
    )
).dt.total_seconds()
pings_exp["scheduled_time_adj"] = (
    pings_exp["_time_adj"] - pings_exp["lateness"] * 60
)

def gtfs_time_to_sec(t):
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

stop_times_all["sched_sec"] = (
    stop_times_all["arrival_time"].apply(gtfs_time_to_sec).astype(float)
)

# every scheduled stop for the runs actually seen in the pings
exp_run_keys = ["service_date", "train_number", "trip_id"]
needed_runs_exp = pings_exp[
    exp_run_keys + ["source_gtfs_date", "line", "direction_id"]
].drop_duplicates()
spine_exp = needed_runs_exp.merge(
    stop_times_all[[
        "trip_id", "source_gtfs_date", "stop_id",
        "stop_sequence", "sched_sec",
    ]],
    on = ["trip_id", "source_gtfs_date"], how = "inner",
)
print(
    f"{len(spine_exp):,} scheduled stop visits across "
    f"{len(needed_runs_exp):,} runs"
)

22,851,788 scheduled stop visits across 1,544,003 runs


In [6]:
spine_exp_sorted = spine_exp.sort_values("sched_sec").reset_index(drop = True)
pings_exp_sorted = (
    pings_exp.sort_values("scheduled_time_adj").reset_index(drop = True)
)

# carry forward each stop's last reported lateness (locf)
exposure = pd.merge_asof(
    spine_exp_sorted,
    pings_exp_sorted[exp_run_keys + ["scheduled_time_adj", "lateness"]],
    left_on = "sched_sec", right_on = "scheduled_time_adj",
    by = exp_run_keys, direction = "backward",
)
n_defaulted = exposure["lateness"].isna().sum()
print(f"{len(exposure):,} scheduled stop visits; {n_defaulted:,} "
      f"({n_defaulted / len(exposure) * 100:.1f}%) had no ping yet, set to 0")
exposure["lateness_locf"] = exposure["lateness"].fillna(0)

dir_label_lookup = {
    (row["line"], str(row["direction_id"])): row["real_direction"]
    for _, row in line_dirs.iterrows()
}
exposure["real_direction"] = [
    dir_label_lookup.get((line, str(did)))
    for line, did in zip(exposure["line"], exposure["direction_id"])
]

# mean delay + ping count per (line, stop, direction)
_grp = (
    exposure.dropna(subset = ["real_direction"])
    .groupby(
        ["line", "stop_id", "real_direction"], observed = True
    )["lateness_locf"]
)
mean_delay_lookup = _grp.mean()
count_delay_lookup = _grp.count()

22,851,788 scheduled stop visits; 1,987,778 (8.7%) had no ping yet, set to 0


## 4. Finding: Amtrak ownership + distance regression (with line fixed effects)

Weighted least squares on hop-level absolute lateness, weighted by underlying observation count,
cluster-robust standard errors by line, with line dummies included so the distance effect is identified
from within-line variation in `stop_position` only (not from longer lines simply having more stops and
different baseline reliability). Includes both `amtrak_x_inbound` and `stop_position_x_inbound`
interactions, since both the Amtrak effect and the distance effect differ sharply by direction.

In [7]:
# one row per consecutive stop pair (a "hop")
# on each line/direction
hop_rows = []
for _, r in line_dirs.iterrows():
    ordered_stops = r["ordered_stops"]
    for frm, to in zip(ordered_stops[:-1], ordered_stops[1:]):
        key = (r["line"], to, r["real_direction"])
        frm_amtrak = amtrak_stop_flags.get((r["line"], frm), False)
        to_amtrak = amtrak_stop_flags.get((r["line"], to), False)
        hop_rows.append({
            "line": r["line"],
            "real_direction": r["real_direction"],
            "stop_id_from": frm,
            "stop_id": to,
            "is_amtrak": frm_amtrak and to_amtrak,
            "stop_position": stop_position_lookup.get(
                (r["line"], to), np.nan
            ),
            "mean_abs_delay": mean_delay_lookup.get(key, np.nan),
            "n_obs_abs": count_delay_lookup.get(key, np.nan),
        })
hops = pd.DataFrame(hop_rows)
hops["is_amtrak"] = hops["is_amtrak"].astype(int)
hops["is_inbound"] = (hops["real_direction"] == "Inbound").astype(int)
hops["amtrak_x_inbound"] = hops["is_amtrak"] * hops["is_inbound"]

d = hops.dropna(
    subset = ["mean_abs_delay", "n_obs_abs", "stop_position"]
).copy()
d["stop_position_x_inbound"] = d["stop_position"] * d["is_inbound"]

# build the full design matrix: amtrak, distance,
# direction, their interactions, and line dummies
line_dummies = pd.get_dummies(d["line"], prefix = "line", drop_first = True)
X = sm.add_constant(pd.concat([
    d[[
        "is_amtrak", "stop_position", "is_inbound",
        "amtrak_x_inbound", "stop_position_x_inbound",
    ]],
    line_dummies,
], axis = 1)).astype(float)

model = sm.WLS(d["mean_abs_delay"], X, weights = d["n_obs_abs"]).fit(
    cov_type = "cluster", cov_kwds = {"groups": d["line"]}
)
print(
    f"n={len(d)} hops, weighted by obs count, "
    f"cluster-robust SE by line, with line fixed effects"
)

implied_inbound_amtrak = (
    model.params["is_amtrak"] + model.params["amtrak_x_inbound"]
)
print(
    f"\nAmtrak effect outbound: {model.params['is_amtrak']:.4f} min, "
    f"p={model.pvalues['is_amtrak']:.4f}"
)
print(f"Amtrak effect inbound:  {implied_inbound_amtrak:.4f} min, "
      f"interaction p={model.pvalues['amtrak_x_inbound']:.4f}")

implied_inbound_dist = (
    model.params["stop_position"] + model.params["stop_position_x_inbound"]
)
print(
    f"\nOutbound distance effect: "
    f"{model.params['stop_position']:.4f} min/stop "
    f"({model.params['stop_position']*60:.2f} sec/stop), "
    f"p={model.pvalues['stop_position']:.4f}"
)
print(
    f"Inbound distance effect:  "
    f"{implied_inbound_dist:.4f} min/stop "
    f"({implied_inbound_dist*60:.2f} sec/stop), "
    f"interaction p={model.pvalues['stop_position_x_inbound']:.4f}"
)

n=400 hops, weighted by obs count, cluster-robust SE by line, with line fixed effects

Amtrak effect outbound: 1.0887 min, p=0.0027
Amtrak effect inbound:  0.4922 min, interaction p=0.0591

Outbound distance effect: 0.0986 min/stop (5.91 sec/stop), p=0.0000
Inbound distance effect:  -0.0424 min/stop (-2.54 sec/stop), interaction p=0.0000


### 4b. Boundary-crossing effect: entrance/exit vs. fully-inside

*Report Table 2 ("Time lost per segment, by segment type") and Appendix C.2/C.3's
boundary-significance numbers.*

Section 4 above looks at mean absolute lateness at a stop. This looks at the within-run, stop-to-stop change in lateness (`delta_lateness = lateness_locf -
previous_stop's lateness_locf`, computed via `groupby(run).shift(1)` on sorted travel order `exposure`).

How much time is gained or lost crossing one specific hop. Every hop is tagged by whether it enters Amtrak-owned track, exits it, stays fully inside it, or stays fully inside SEPTA track

In [8]:
import os as _os

_deltas_cache_path = f"{BASEPATH}/11_deltas_with_hop_type.parquet"

if _os.path.exists(_deltas_cache_path):
    deltas = pd.read_parquet(_deltas_cache_path)
else:
    # sort each run into stop order, then grab
    # each row's previous stop + lateness
    exposure_sorted = (
        exposure.sort_values(exp_run_keys + ["stop_sequence"])
        .reset_index(drop = True)
    )
    _grp_run = exposure_sorted.groupby(exp_run_keys, sort = False)
    exposure_sorted["prev_stop_id"] = _grp_run["stop_id"].shift(1)
    exposure_sorted["prev_lateness_locf"] = _grp_run["lateness_locf"].shift(1)
    exposure_sorted["same_run_prev"] = _grp_run.cumcount() > 0
    exposure_sorted["delta_lateness"] = (
        exposure_sorted["lateness_locf"]
        - exposure_sorted["prev_lateness_locf"]
    )

    # drop each run's first stop -- nothing to
    # compare it against
    deltas = exposure_sorted[exposure_sorted["same_run_prev"]].dropna(
        subset = ["prev_stop_id", "delta_lateness", "real_direction"]
    ).copy()
    print(f"{len(deltas):,} within-run hops")

    deltas["prev_is_amtrak"] = [
        amtrak_stop_flags.get((line, sid), np.nan)
        for line, sid in zip(deltas["line"], deltas["prev_stop_id"])
    ]
    deltas["curr_is_amtrak"] = [
        amtrak_stop_flags.get((line, sid), np.nan)
        for line, sid in zip(deltas["line"], deltas["stop_id"])
    ]

    # classify each hop: entering, exiting, or
    # staying within amtrak-owned track
    _prev_na = deltas["prev_is_amtrak"].isna()
    _curr_na = deltas["curr_is_amtrak"].isna()
    _prev_bool = deltas["prev_is_amtrak"].fillna(False).astype(bool)
    _curr_bool = deltas["curr_is_amtrak"].fillna(False).astype(bool)

    # stops with no match get "no_data"
    deltas["hop_type"] = np.select(
        [
            _prev_na | _curr_na,
            (~_prev_bool) & _curr_bool,
            _prev_bool & (~_curr_bool),
            _prev_bool & _curr_bool,
        ],
        ["no_data", "entering_amtrak", "exiting_amtrak", "within_amtrak"],
        default = "within_septa",
    )
    # swap "no_data" for NA
    deltas.loc[deltas["hop_type"] == "no_data", "hop_type"] = np.nan

    deltas["stop_position"] = [
        stop_position_lookup.get((line, sid), np.nan)
        for line, sid in zip(deltas["line"], deltas["stop_id"])
    ]
    deltas["is_inbound"] = (deltas["real_direction"] == "Inbound").astype(int)

    deltas.to_parquet(_deltas_cache_path, index = False)

# mean/median incremental delay, by hop type
boundary_table = (
    deltas.groupby("hop_type")["delta_lateness"]
    .agg(["mean", "median", "count"])
)
boundary_table.columns = ["mean_delta_min", "median_delta_min", "n_stop_visits"]
print("\nIncremental delay by hop type, all runs:")
print(boundary_table)

boundary_table.to_parquet(f"{BASEPATH}/11_boundary_hop_table.parquet")

20,220,260 within-run hops



Incremental delay by hop type, all runs:
                 mean_delta_min  median_delta_min  n_stop_visits
hop_type                                                        
entering_amtrak        0.854677               0.0         529825
exiting_amtrak         0.735511               0.0         421106
within_amtrak          0.124508               0.0        5925611
within_septa           0.154353               0.0       12886549


Significance: hop-type regression (`within_septa` as reference category), weighted by observation count,
cluster-robust SEs by line, controlling for `stop_position` and `is_inbound`.
*Appendix C.2/C.3's boundary-crossing coefficients*

In [9]:
# one row per distinct hop -- avg delta and obs count feed the reg below
deltas_valid = deltas.dropna(
    subset = ["hop_type", "stop_position", "line"]
).copy()
hop_agg = (
    deltas_valid
    .groupby(
        ["line", "prev_stop_id", "stop_id", "real_direction", "hop_type"],
        observed = True,
    )["delta_lateness"]
    .agg(["mean", "count"]).reset_index()
)
hop_agg = hop_agg.rename(columns = {"mean": "mean_delta", "count": "n_obs"})
hop_agg["stop_position"] = [
    stop_position_lookup.get((l, s), np.nan)
    for l, s in zip(hop_agg["line"], hop_agg["stop_id"])
]
hop_agg["is_inbound"] = (hop_agg["real_direction"] == "Inbound").astype(int)
hop_agg = hop_agg.dropna(subset = ["stop_position"])
print(f"n={len(hop_agg)} rows across all 4 hop types")

type_dummies = pd.get_dummies(
    hop_agg["hop_type"], prefix = "type", drop_first = False
).astype(float)
# reference category
type_dummies = type_dummies.drop(columns = ["type_within_septa"])
Xsig = sm.add_constant(pd.concat(
    [type_dummies, hop_agg[["stop_position", "is_inbound"]]],
    axis = 1,
))
model_sig = sm.WLS(hop_agg["mean_delta"], Xsig, weights = hop_agg["n_obs"]).fit(
    cov_type = "cluster", cov_kwds = {"groups": hop_agg["line"]}
)
print("\nWithout line fixed effects:")
print(model_sig.summary().tables[1])

# same model + a dummy per line, so the hop-type
# effect isn't just picking up which lines run late
line_dummies3 = pd.get_dummies(
    hop_agg["line"], prefix = "line", drop_first = True
).astype(float)
Xsig_fe = pd.concat([Xsig.drop(columns = ["const"]), line_dummies3], axis = 1)
Xsig_fe = sm.add_constant(Xsig_fe)
model_sig_fe = sm.WLS(
    hop_agg["mean_delta"], Xsig_fe, weights = hop_agg["n_obs"]
).fit(
    cov_type = "cluster", cov_kwds = {"groups": hop_agg["line"]}
)
print("\nWith line fixed effects:")
hop_types = [
    "type_entering_amtrak", "type_exiting_amtrak", "type_within_amtrak",
]
for col in hop_types:
    print(
        f"{col}: coef={model_sig_fe.params[col]:.4f}, "
        f"se={model_sig_fe.bse[col]:.4f}, "
        f"p={model_sig_fe.pvalues[col]:.4f}"
    )

hop_type_names = ["entering_amtrak", "exiting_amtrak", "within_amtrak"]
boundary_significance = pd.DataFrame({
    "hop_type": hop_type_names,
    "coef_no_fe": [model_sig.params[f"type_{t}"] for t in hop_type_names],
    "p_no_fe": [model_sig.pvalues[f"type_{t}"] for t in hop_type_names],
    "coef_line_fe": [
        model_sig_fe.params[f"type_{t}"] for t in hop_type_names
    ],
    "p_line_fe": [
        model_sig_fe.pvalues[f"type_{t}"] for t in hop_type_names
    ],
})
boundary_significance.to_parquet(
    f"{BASEPATH}/11_boundary_significance.parquet", index = False
)

n=1449 rows across all 4 hop types

Without line fixed effects:
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.2595      0.039      6.723      0.000       0.184       0.335
type_entering_amtrak     0.6446      0.118      5.453      0.000       0.413       0.876
type_exiting_amtrak      0.5447      0.358      1.523      0.128      -0.156       1.246
type_within_amtrak      -0.0099      0.025     -0.389      0.697      -0.060       0.040
stop_position           -0.0080      0.003     -2.828      0.005      -0.014      -0.002
is_inbound              -0.0703      0.021     -3.349      0.001      -0.111      -0.029

With line fixed effects:
type_entering_amtrak: coef=0.7759, se=0.1137, p=0.0000
type_exiting_amtrak: coef=0.6746, se=0.3671, p=0.0662
type_within_amtrak: coef=0.1459, se=0.0601, p=0.0153


### 4c. Cache per-hop data for report figures

Consolidates both quantities used across report figures -- mean absolute lateness (level, from
Section 4's `mean_delay_lookup`) and mean incremental delay (accumulated-per-hop delta, from Section 4b's
`deltas`) -- into flat table. Keyed by `(line, real_direction, stop_id)` for every
line-direction (not just Outbound/the 6 Amtrak lines).

*`12_figures.ipynb` loads this for plotting*

In [10]:
# combine the level (mean_abs_delay) and delta
# (mean_delta) into one row per stop
mean_delta_by_stop = deltas.dropna(subset = ["stop_position"]).groupby(
    ["line", "stop_id", "real_direction"], observed = True
)["delta_lateness"].agg(["mean", "count"])
mean_delta_by_stop.columns = ["mean_delta", "n_obs_delta"]

hop_flat_rows = []
for _, r in line_dirs.iterrows():
    ordered_stops = r["ordered_stops"]
    for i, s in enumerate(ordered_stops):
        if i == 0:
            continue  # no incoming hop for the first stop of a trip
        key = (r["line"], s, r["real_direction"])
        hop_flat_rows.append({
            "line": r["line"],
            "real_direction": r["real_direction"],
            "stop_id": s,
            "stop_name": stn_name_lookup.get(s, s),
            "stop_position": stop_position_lookup.get((r["line"], s), np.nan),
            "is_amtrak": amtrak_stop_flags.get((r["line"], s), False),
            "mean_abs_delay": mean_delay_lookup.get(key, np.nan),
            "n_obs_abs": count_delay_lookup.get(key, np.nan),
            "mean_delta": mean_delta_by_stop["mean_delta"].get(key, np.nan),
            "n_obs_delta": mean_delta_by_stop["n_obs_delta"].get(key, np.nan),
        })
hops_flat = pd.DataFrame(hop_flat_rows)
hops_flat.to_parquet(f"{BASEPATH}/11_hops_flat.parquet", index = False)
print(hops_flat.head())

              line real_direction stop_id            stop_name  stop_position  \
0  Paoli/Thorndale       Outbound   90006            Jefferson              1   
1  Paoli/Thorndale       Outbound   90005     Suburban Station              2   
2  Paoli/Thorndale       Outbound   90004  30th Street Station              3   
3  Paoli/Thorndale       Outbound   90522            Overbrook              4   
4  Paoli/Thorndale       Outbound   90521               Merion              5   

   is_amtrak  mean_abs_delay  n_obs_abs  mean_delta  n_obs_delta  
0      False        2.988941      59135    2.409990        55277  
1      False        1.530785      83011   -0.852745        59244  
2       True        2.797461      83031    1.260122        83011  
3       True        4.552308      73392    1.723711        73376  
4       True        4.735393      72534    0.160297        72534  


### 4d. Cache route/station geometry for the map figure

*Saved and later loaded for `12_figures.ipynb`*

In [11]:
# one line geometry per full route, and one
# short segment per hop
line_route_rows = []
for _, r in line_dirs.iterrows():
    geom = LineString([
        stn_points.loc[s, "geometry"] for s in r["ordered_stops"]
    ])
    line_route_rows.append({
        "line": r["line"],
        "real_direction": r["real_direction"],
        "geometry": geom,
    })
gdf_routes = gpd.GeoDataFrame(
    line_route_rows, geometry = "geometry", crs = PROJECTED_CRS
)

hop_geoms = []
for _, r in line_dirs.iterrows():
    ordered_stops = r["ordered_stops"]
    for frm, to in zip(ordered_stops[:-1], ordered_stops[1:]):
        hop_geoms.append(LineString([
            stn_points.loc[frm, "geometry"],
            stn_points.loc[to, "geometry"],
        ]))
gdf_hops_geo = gpd.GeoDataFrame(
    hops.copy(), geometry = hop_geoms, crs = PROJECTED_CRS
)

# cache geometry tables for 12_figures.ipynb --
# no plotting here
gdf_hops_geo[
    ["line", "real_direction", "is_amtrak", "mean_abs_delay", "geometry"]
].to_parquet(f"{BASEPATH}/11_hop_geometries.parquet")
stn_points.reset_index()[["stop_id", "geometry"]].to_parquet(
    f"{BASEPATH}/11_station_points.parquet"
)

# terminus stop per (line, direction) -- used
# for map annotation labels in 12_figures.ipynb
termini = line_dirs.copy()
termini["terminus_stop_id"] = termini["ordered_stops"].apply(lambda s: s[-1])
termini[["line", "real_direction", "terminus_stop_id"]].to_parquet(
    f"{BASEPATH}/11_termini.parquet", index = False
)

## Summary

Amtrak-owned track runs later than SEPTA-owned track, but the effect is concentrated in the Outbound
direction and nearly disappears Inbound (distance and line FE controlled regression).